In [6]:
!optimum-cli export onnx --model ./finetuned_whisper_model_akan/saved_model --library-name transformers --task automatic-speech-recognition-with-past ./whisper_tiny_onnx

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}
C:\Users\KASA\Downloads\Kasa\venv\Lib\site-packages\transformers\models\whisper\modeling_whisper.py:1159: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if input_features.shape[-1] != expected_seq_length:
C:\Users\KASA\Downloads\Kasa\venv\Lib\site-packages\transformers\models\whisper\modeling_whisper.py:338: TracerWarning: Converting a tensor to a Python boolean might cau

In [8]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from optimum.onnxruntime import ORTModelForSpeechSeq2Seq

MODEL_NAME = "./finetuned_whisper_model_akan/saved_processor"  # or your depth-reduced checkpoint path
EXPORT_PATH = "./whisper_tiny_onnx"


processor = WhisperProcessor.from_pretrained(MODEL_NAME)
processor.save_pretrained(EXPORT_PATH)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


[]

In [9]:
from optimum.onnxruntime import ORTModelForSpeechSeq2Seq
from transformers import WhisperProcessor

model = ORTModelForSpeechSeq2Seq.from_pretrained("./whisper_tiny_onnx")
processor = WhisperProcessor.from_pretrained("./whisper_tiny_onnx")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [ ]:
from optimum.onnxruntime import ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

QUANT_PATH = "./whisper_kasa_onnx_int8"

# dynamic INT8 quantization — but via ONNX Runtime's quantizer,
# which (unlike PyTorch's) does quantize conv/attention ops
qconfig = AutoQuantizationConfig.arm64(is_static=False, per_channel=True)

for component in ["encoder_model", "decoder_model", "decoder_with_past_model"]:
    quantizer = ORTQuantizer.from_pretrained(EXPORT_PATH, file_name=f"{component}.onnx")
    quantizer.quantize(save_dir=QUANT_PATH, quantization_config=qconfig)

processor.save_pretrained(QUANT_PATH)

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This 

[]

In [21]:
from optimum.onnxruntime import ORTModelForSpeechSeq2Seq

quantized_model = ORTModelForSpeechSeq2Seq.from_pretrained(
    QUANT_PATH,
    encoder_file_name="encoder_model_quantized.onnx",
    decoder_file_name="decoder_model_quantized.onnx",
    decoder_with_past_file_name="decoder_with_past_model_quantized.onnx",
)

In [ ]:
from datasets import load_dataset, Audio

dataset = load_dataset(
    "Kennethdot/Ghana_English-Twi_Code-switching_Speech",
)

dataset = dataset.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)


In [ ]:
def transcribe_from_dataset(dataset_sample, whisper_model, max_new_tokens=128):
    input_features = processor.feature_extractor(
        dataset_sample["array"],
        sampling_rate=dataset_sample["sampling_rate"],
        return_tensors="pt"
    ).input_features

    predicted_ids = whisper_model.generate(
        input_features,
        max_new_tokens=max_new_tokens,
        task="transcribe",
        forced_decoder_ids=None
    )
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)
    return transcription[0].strip()

In [14]:
import re

def normalize_cs(text):
    text = text.lower()

    # keep English letters + Twi chars
    text = re.sub(r"[^a-z0-9ɔɛ\s']", "", text)

    # normalize apostrophes (optional)
    text = re.sub(r"'", "", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [17]:
from evaluate import load as metrics_loader
wer_metric = metrics_loader("wer")

def get_wer(references, predictions, normalize=True, verbose=True):
  rs = references
  ps = predictions
  if normalize:
    ps = [normalize_cs(x) for x in predictions]
    rs = [normalize_cs(x) for x in references]
  if verbose:
    for r, p in zip(rs, ps):
      print(r)
      print(p)
      print()

  return wer_metric.compute(references=rs, predictions=ps)

In [20]:
from transformers import GenerationConfig

fresh_gen_config = GenerationConfig.from_pretrained("openai/whisper-small")
quantized_model.generation_config = fresh_gen_config
quantized_model.generation_config.save_pretrained(QUANT_PATH)

generation_config.json: 0.00B [00:00, ?B/s]

c:\Users\KASA\Downloads\Kasa\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\KASA\.cache\huggingface\hub\models--openai--whisper-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [22]:
normalize_for_wer_calc = True #@param{type: 'boolean'}

print('number of test examples to process:', len(dataset['test']))

predictions = []
finetuned_predictions = []
references  = []

for idx in range(len(dataset['test'])):
  print('inference on example:', idx)
  sample = dataset['test'][idx]["audio"]
  predictions.append(transcribe_from_dataset(sample, quantized_model))
  references.append(dataset['test'][idx]['transcript'])

number of test examples to process: 1731
inference on example: 0
inference on example: 1
inference on example: 2
inference on example: 3
inference on example: 4
inference on example: 5
inference on example: 6
inference on example: 7
inference on example: 8
inference on example: 9
inference on example: 10
inference on example: 11
inference on example: 12
inference on example: 13
inference on example: 14
inference on example: 15
inference on example: 16
inference on example: 17
inference on example: 18
inference on example: 19
inference on example: 20
inference on example: 21
inference on example: 22
inference on example: 23
inference on example: 24
inference on example: 25
inference on example: 26
inference on example: 27
inference on example: 28
inference on example: 29
inference on example: 30
inference on example: 31
inference on example: 32
inference on example: 33
inference on example: 34
inference on example: 35
inference on example: 36
inference on example: 37
inference on exampl

KeyboardInterrupt: 

In [24]:
import os

def get_onnx_model_size_mb(model_path):
    """Sum the size of all .onnx files in the model directory."""
    total_bytes = 0
    for fname in os.listdir(model_path):
        if fname.endswith(".onnx"):
            fpath = os.path.join(model_path, fname)
            total_bytes += os.path.getsize(fpath)
    return total_bytes / (1024 * 1024)

quantized_size_mb = get_onnx_model_size_mb(QUANT_PATH)
print(f"Quantized model size: {quantized_size_mb:.2f} MB")

Quantized model size: 66.79 MB


In [25]:
unquantized_size_mb = get_onnx_model_size_mb(EXPORT_PATH)
print(f"Unquantized ONNX model size: {unquantized_size_mb:.2f} MB")
print(f"Compression ratio: {unquantized_size_mb / quantized_size_mb:.2f}x")

Unquantized ONNX model size: 365.62 MB
Compression ratio: 5.47x


In [26]:
finetuned_wer = get_wer(references=references, predictions=predictions, normalize=normalize_for_wer_calc, verbose=False)

print(f'FINETUNED WER: {finetuned_wer}')

FINETUNED WER: 0.2482775590551181


In [27]:
for transcript, prediction in zip(references, predictions):
  print('REFERENCE:', transcript)
  print('PREDICTION:', prediction)
  print()

REFERENCE: “Sɛ wopɛ a, yɛbɛtumi anante, it’s not that far.”
PREDICTION: Sɛ wopɛ a, yɛbɛtumi anante, it’s not that far.

REFERENCE: Ɛyɛ den oo, but I’ll try.
PREDICTION: Ɛyɛ den oo, but I’ll try.

REFERENCE: Cough syrup no yɛ dɛ dodo.
PREDICTION: Cough syrup no yɛ dɛ dodo.

REFERENCE: Nyame ne hene daa, God is good.
PREDICTION: Nyame ne hene daa, God is good.

REFERENCE: Mepakyew, boa me, it is an emergency.
PREDICTION: Mepakyew, boa me, it’s anɛbɛ nsuo.

REFERENCE: “Wo ho te sɛn? You’ve been quiet all day.”
PREDICTION: “Wo ho te sɛn? You’ve been quiet all day.”

REFERENCE: Ma yɛnkɔ y’anim kakra, let’s skip this place.
PREDICTION: Ma yɛnkɔ y’anim kakra, let’s skip this place.

REFERENCE: She wore a white cloth because na n’ani agye.
PREDICTION: She wore a white cloth because na n’ani agye.

REFERENCE: Menni cash, can I pay with my phone?
PREDICTION: Menni cash, can I pay with my phone no?

REFERENCE: Anka mepɛ waakye, but it is finished.
PREDICTION: Anka mepɛ waakye, but it is finished.

### Whisper.cpp

In [9]:
!git clone https://github.com/openai/whisper openai-whisper-repo

fatal: destination path 'openai-whisper-repo' already exists and is not an empty directory.


In [33]:
!python whisper.cpp/models/convert-h5-to-ggml.py ./finetuned_whisper_model_akan ./openai-whisper-repo ./whisper.cpp/models

model.encoder.conv1.weight  ->  encoder.conv1.weight
encoder.conv1.weight 3 (384, 80, 3)
model.encoder.conv1.bias  ->  encoder.conv1.bias
  Reshaped variable:  encoder.conv1.bias  to shape:  (384, 1)
encoder.conv1.bias 2 (384, 1)
  Converting to float32
model.encoder.conv2.weight  ->  encoder.conv2.weight
encoder.conv2.weight 3 (384, 384, 3)
model.encoder.conv2.bias  ->  encoder.conv2.bias
  Reshaped variable:  encoder.conv2.bias  to shape:  (384, 1)
encoder.conv2.bias 2 (384, 1)
  Converting to float32
model.encoder.embed_positions.weight  ->  encoder.positional_embedding
encoder.positional_embedding 2 (1500, 384)
  Converting to float32
model.encoder.layers.0.self_attn.k_proj.weight  ->  encoder.blocks.0.attn.key.weight
encoder.blocks.0.attn.key.weight 2 (384, 384)
model.encoder.layers.0.self_attn.v_proj.weight  ->  encoder.blocks.0.attn.value.weight
encoder.blocks.0.attn.value.weight 2 (384, 384)
model.encoder.layers.0.self_attn.v_proj.bias  ->  encoder.blocks.0.attn.value.bias
enco

In [3]:
import os
print(os.listdir("./whisper.cpp/models"))

['.gitignore', 'convert-h5-to-coreml.py', 'convert-h5-to-ggml.py', 'convert-parakeet-to-ggml.py', 'convert-pt-to-ggml.py', 'convert-silero-vad-to-ggml.py', 'convert-whisper-to-coreml.py', 'convert-whisper-to-openvino.py', 'download-coreml-model.sh', 'download-ggml-model.cmd', 'download-ggml-model.sh', 'download-vad-model.cmd', 'download-vad-model.sh', 'for-tests-ggml-base.bin', 'for-tests-ggml-base.en.bin', 'for-tests-ggml-large.bin', 'for-tests-ggml-medium.bin', 'for-tests-ggml-medium.en.bin', 'for-tests-ggml-parakeet-tdt-bad-nfft0.bin', 'for-tests-ggml-parakeet-tdt.bin', 'for-tests-ggml-small.bin', 'for-tests-ggml-small.en.bin', 'for-tests-ggml-tiny.bin', 'for-tests-ggml-tiny.en.bin', 'for-tests-silero-v6.2.0-ggml.bin', 'generate-coreml-interface.sh', 'generate-coreml-model.sh', 'generate-parakeet-test-model.py', 'ggml_to_pt.py', 'README.md', 'requirements-coreml.txt', 'requirements-openvino.txt', 'requirements-parakeet.txt']


In [4]:
size_mb = os.path.getsize("./models_export/ggml-model.bin") / (1024 * 1024)
print(f"GGML model size: {size_mb:.2f} MB")

GGML model size: 74.09 MB


In [42]:
!winget install Kitware.CMake

Found CMake [Kitware.CMake] Version 4.4.2
This application is licensed to you by its owner.
Microsoft is not responsible for, nor does it grant any licenses to, third-party packages.
Successfully verified installer hash
Starting package install...


In [5]:
!cmake --version

cmake version 4.4.2

CMake suite maintained and supported by Kitware (kitware.com/cmake).


In [6]:
!cmake -S whisper.cpp -B whisper.cpp/build -DCMAKE_BUILD_TYPE=Release
!cmake --build whisper.cpp/build --config Release

-- Building for: Visual Studio 18 2026
-- The C compiler identification is MSVC 19.51.36252.0
-- The CXX compiler identification is MSVC 19.51.36252.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: C:/Program Files (x86)/Microsoft Visual Studio/18/BuildTools/VC/Tools/MSVC/14.51.36231/bin/Hostx64/x64/cl.exe - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: C:/Program Files (x86)/Microsoft Visual Studio/18/BuildTools/VC/Tools/MSVC/14.51.36231/bin/Hostx64/x64/cl.exe - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found Git: C:/Program Files/Git/cmd/git.exe (found version "2.54.0.windows.1")
-- The ASM compiler identification is MSVC
-- Found assembler: C:/Program Files (x86)/Microsoft Visual Studio/18/BuildTools/VC/Tools/MSVC/14.51.36231/bin/Hostx64/x

CMake Warning (deprecated) at CMakeLists.txt:1 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.
This warning is for project developers.  Use -Wno-author or -Wno-deprecated
to suppress it.

CMake Warning (policy) at C:/Program Files/CMake/share/cmake-4.4/Modules/CMakeDetermineASMCompiler.cmake:237 (cmake_policy):
  Policy CMP0194 is not set: MSVC is not an assembler for language ASM.  Run
  "cmake --help-policy CMP0194" for policy details.  Use the cmake_policy
  command to set the policy and suppress this warning.
Call Stack (most recent call first):
  ggml/CMakeLists.txt:3 (project)
This warning is for project developers.  Use -Wno-author or -Wno-policy to
suppress it.



MSBuild version 18.8.2+ce25c0108 for .NET Framework

  1>Checking Build System
  Building Custom Rule C:/Users/KASA/Downloads/Kasa/quant/whisper.cpp/examples/deprecation-warning/CMakeLists.txt
  deprecation-warning.cpp
  bench.vcxproj -> C:\Users\KASA\Downloads\Kasa\quant\whisper.cpp\build\bin\Release\bench.exe
  Building Custom Rule C:/Users/KASA/Downloads/Kasa/quant/whisper.cpp/ggml/src/CMakeLists.txt
  ggml.c
  ggml.cpp
  ggml-alloc.c
  ggml-quants.c
  Generating Code...
  ggml-backend.cpp
  ggml-backend-meta.cpp
  ggml-opt.cpp
  ggml-threading.cpp
  gguf.cpp
  Generating Code...
     Creating library C:/Users/KASA/Downloads/Kasa/quant/whisper.cpp/build/ggml/src/Release/ggml-base.lib and object C:/Users/KASA/Downloads/Kasa/quant/whisper.cpp/build/ggml/src/Release/ggml-base.exp
  ggml-base.vcxproj -> C:\Users\KASA\Downloads\Kasa\quant\whisper.cpp\build\bin\Release\ggml-base.dll
  Building Custom Rule C:/Users/KASA/Downloads/Kasa/quant/whisper.cpp/ggml/src/CMakeLists.txt
  ggml-cpu.c


In [7]:
for root, dirs, files in os.walk("whisper.cpp/build"):
    for f in files:
        if "quantize" in f.lower():
            print(os.path.join(root, f))

whisper.cpp/build\bin\Release\parakeet-quantize.exe
whisper.cpp/build\bin\Release\whisper-quantize.exe
whisper.cpp/build\examples\parakeet-quantize\parakeet-quantize.vcxproj
whisper.cpp/build\examples\parakeet-quantize\parakeet-quantize.vcxproj.filters
whisper.cpp/build\examples\parakeet-quantize\parakeet-quantize.dir\Release\parakeet-quantize.exe.recipe
whisper.cpp/build\examples\parakeet-quantize\parakeet-quantize.dir\Release\parakeet-quantize.obj
whisper.cpp/build\examples\parakeet-quantize\parakeet-quantize.dir\Release\parakeet.AF828641.tlog\parakeet-quantize.lastbuildstate
whisper.cpp/build\examples\quantize\whisper-quantize.vcxproj
whisper.cpp/build\examples\quantize\whisper-quantize.vcxproj.filters
whisper.cpp/build\examples\quantize\whisper-quantize.dir\Release\quantize.obj
whisper.cpp/build\examples\quantize\whisper-quantize.dir\Release\whisper-quantize.exe.recipe
whisper.cpp/build\examples\quantize\whisper-quantize.dir\Release\whisper-quantize.tlog\whisper-quantize.lastbuilds

In [17]:
import os

cwd = os.getcwd()
quantize_bin = os.path.join(cwd, "whisper.cpp", "build", "bin", "Release", "whisper-quantize.exe")
input_model = os.path.join(cwd, "models_export", "ggml-model.bin")
output_model = os.path.join(cwd, "models_export", "ggml-model_q_8.bin")

print(quantize_bin)
print(os.path.exists(quantize_bin))

C:\Users\KASA\Downloads\Kasa\quant\whisper.cpp\build\bin\Release\whisper-quantize.exe
True


In [18]:
import subprocess

result = subprocess.run(
    [quantize_bin, input_model, output_model, "q8_0"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

whisper_model_quantize: loading model from 'C:\Users\KASA\Downloads\Kasa\quant\models_export\ggml-model.bin'
                                            encoder.conv1.weight - [    3,    80,   384], type =    f16 size =    0.176 MB
                                              encoder.conv1.bias - [    1,   384,     1], type =    f32 size =    0.001 MB
                                            encoder.conv2.weight - [    3,   384,   384], type =    f16 size =    0.844 MB
                                              encoder.conv2.bias - [    1,   384,     1], type =    f32 size =    0.001 MB
                                    encoder.positional_embedding - [  384,  1500,     1], type =    f32 size =    2.197 MB
                                encoder.blocks.0.attn.key.weight - [  384,   384,     1], type =    f16 size =     0.56 MB ->     0.15 MB
                              encoder.blocks.0.attn.value.weight - [  384,   384,     1], type =    f16 size =     0.56 MB ->     0.15 MB


In [24]:
dataset['test'][0]

{'speaker_id': '35b8',
 'age_range': '18-24',
 'gender': 'male',
 'prompt_set': 'standard',
 'transcript': '“Sɛ wopɛ a, yɛbɛtumi anante, it’s not that far.”',
 'duration': 4.853499889373779,
 'split': 'test',
 'audio': {'path': '35b8_std_1_20260225T213853Z.ogg',
  'array': array([ 2.85863644e-06, -3.76813114e-06, -1.07599772e-05, ...,
          1.29173987e-03,  1.86404213e-04,  3.27933254e-03]),
  'sampling_rate': 16000},
 'file_name': None,
 'error': None}

In [26]:
import soundfile as sf
import json
references = {}
os.makedirs("wavs", exist_ok=True)

for i, sample in enumerate(dataset["test"]):
    audio = sample["audio"]
    # use the audio path as a stable id, falling back to index
    file_id = os.path.splitext(os.path.basename(audio["path"]))[0] if audio["path"] else f"{i:05d}"
    sf.write(f"wavs/{file_id}.wav", audio["array"], audio["sampling_rate"])
    references[file_id] = sample["transcript"]

with open("references.json", "w", encoding="utf-8") as f:
    json.dump(references, f, ensure_ascii=False, indent=2)

In [40]:
import subprocess
import os

binary_path = r"whisper.cpp\build\bin\Release\whisper-cli.exe"
model_path = "./models_export/ggml-model_q8.bin"

print("binary exists:", os.path.exists(binary_path))
print("model exists:", os.path.exists(model_path))

binary exists: True
model exists: True


In [41]:
f = "wavs/" + os.listdir("wavs")[0]
file_id = os.path.splitext(os.path.basename(f))[0]

os.makedirs("transcripts", exist_ok=True)

result = subprocess.run([
    binary_path,
    "-m", model_path,
    "-f", f,
    "-otxt",
    "-of", f"transcripts/{file_id}"
], capture_output=True, text=True)

print("stdout:", result.stdout)
print("stderr:", result.stderr)
print("return code:", result.returncode)

stdout: 
[00:00:00.500 --> 00:00:30.000]  â€œWo yÉ› sure? Lunch will end soon o.

stderr: whisper_init_from_file_with_params_no_state: loading model from './models_export/ggml-model_q8.bin'
whisper_init_with_params_no_state: use gpu    = 1
whisper_init_with_params_no_state: flash attn = 1
whisper_init_with_params_no_state: gpu_device = 0
whisper_init_with_params_no_state: dtw        = 0
whisper_init_with_params_no_state: devices    = 1
whisper_init_with_params_no_state: backends   = 1
whisper_model_load: loading model
whisper_model_load: n_vocab       = 51865
whisper_model_load: n_audio_ctx   = 1500
whisper_model_load: n_audio_state = 384
whisper_model_load: n_audio_head  = 6
whisper_model_load: n_audio_layer = 4
whisper_model_load: n_text_ctx    = 448
whisper_model_load: n_text_state  = 384
whisper_model_load: n_text_head   = 6
whisper_model_load: n_text_layer  = 4
whisper_model_load: n_mels        = 80
whisper_model_load: ftype         = 7
whisper_model_load: qntvr         = 2
whispe

In [49]:
import subprocess
import os
import time
import json


wav_files = [f for f in os.listdir("wavs") if f.endswith(".wav")]
print(f"Total files: {len(wav_files)}")

results = {}
failed = []
start = time.time()

if os.path.exists("transcripts.json"):
    with open("transcripts.json", encoding="utf-8") as f:
        results = json.load(f)
    print(f"Loaded {len(results)} existing transcripts, will skip those")
else:
    results = {}

for i, fname in enumerate(wav_files):
    file_id = os.path.splitext(fname)[0]
    if file_id in results:
        continue  
    result = subprocess.run([
        binary_path,
        "-m", model_path,
        "-f", f"wavs/{fname}",
        "-nt"  # no timestamps, just plain text to stdout
    ], capture_output=True, text=True, encoding="utf-8", errors="replace")

    if result.returncode != 0:
        failed.append((fname, result.stderr))
        continue

    results[file_id] = result.stdout.strip()

    if (i + 1) % 20 == 0:
        elapsed = time.time() - start
        print(f"{i+1}/{len(wav_files)} done ({elapsed:.1f}s elapsed)")
        # save progress incrementally so a crash doesn't lose everything
        with open("transcripts.json", "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

# final save
with open("transcripts.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\nFinished. {len(results)} succeeded, {len(failed)} failed.")
for fname, err in failed[:5]:
    print(fname, "->", err[:200])

Total files: 1651
Loaded 300 existing transcripts, will skip those
320/1651 done (9.2s elapsed)
340/1651 done (19.1s elapsed)
360/1651 done (28.8s elapsed)
380/1651 done (38.5s elapsed)
400/1651 done (48.2s elapsed)
420/1651 done (58.1s elapsed)
440/1651 done (67.7s elapsed)
460/1651 done (77.3s elapsed)
480/1651 done (87.7s elapsed)
500/1651 done (97.6s elapsed)
520/1651 done (107.3s elapsed)
540/1651 done (117.8s elapsed)
560/1651 done (128.5s elapsed)
580/1651 done (138.5s elapsed)
600/1651 done (148.6s elapsed)
620/1651 done (158.7s elapsed)
640/1651 done (168.8s elapsed)
660/1651 done (178.7s elapsed)
680/1651 done (188.5s elapsed)
700/1651 done (198.6s elapsed)
720/1651 done (208.4s elapsed)
740/1651 done (218.0s elapsed)
760/1651 done (228.0s elapsed)
780/1651 done (237.7s elapsed)
800/1651 done (247.5s elapsed)
820/1651 done (265.2s elapsed)
840/1651 done (289.3s elapsed)
860/1651 done (313.5s elapsed)
880/1651 done (334.2s elapsed)
900/1651 done (357.8s elapsed)
920/1651 done 

In [ ]:
import json
from jiwer import wer, cer

# load references
with open("references.json", encoding="utf-8") as f:
    references = json.load(f)

# load transcripts (single-file version from the updated inference loop)
with open("transcripts.json", encoding="utf-8") as f:
    transcripts = json.load(f)

# only score files that succeeded in both dicts
common_ids = [fid for fid in references if fid in transcripts]
missing = [fid for fid in references if fid not in transcripts]

print(f"Scoring {len(common_ids)} / {len(references)} files")
if missing:
    print(f"Missing {len(missing)} transcripts (failed inference?): {missing[:5]}")

refs = [references[fid] for fid in common_ids]
hyps = [transcripts[fid] for fid in common_ids]

overall_wer = wer(refs, hyps)
overall_cer = cer(refs, hyps)

print(f"Overall WER: {overall_wer:.3f}")
print(f"Overall CER: {overall_cer:.3f}")

Scoring 1651 / 1651 files
Overall WER: 0.081
Overall CER: 0.046


: 